# L23 · 梯度（gradient） — 多元函数（multivariate function）的"最陡上坡"方向，偏导（partial derivative，∂f/∂x）与梯度向量（gradient vector）的计算

**目标**：多变量函数对每个变量分别求导(偏导)，拼成一个向量=梯度。梯度指向函数上升最快的方向。

**为什么对 Aurora 重要**：反向传播（`backward`）计算损失对每个权重的偏导，拼成梯度向量；优化器用这个向量做一步参数更新（`optimizer.step()`）。


## 本课剧情：拿到多方向罗盘

对多变量函数，偏导把其他变量暂时冻住，只测一个方向的斜率。把每个变量的偏导数（derivative）拼成一列，就得到梯度向量——它的方向是函数在当前点上升最快的方向。数值计算用中心差分（central difference，CD）：对每个分量分别施加 ±h 扰动，两侧相减再除以 2h 即得。


## 1. 偏导：固定其他变量，只对一个变量求导

对多元函数 f(x, y)，偏导 ∂f/∂x 的计算：把 y 当常数，只对 x 求导。中心差分把这个操作数值化：给第 i 个分量加减 h，其余分量保持不变，再除以 2h。

**设计原因**：每次只扰动一个维度，保持其他维度不变——这就是偏导数的定义。n 维函数需要 2n 次函数求值（每个维度正负各一次）。神经网络有数百万参数，对每个参数都做两次前向传播代价高不可接受，所以才需要反向传播：一次前向 + 一次后向即可得到所有参数的梯度，计算量与参数量无关。

## 实验入口：用数值变化观察函数

观察 `dfdx` 和 `dfdy` 在点 `(3, 4)` 处的值，确认中心差分结果与解析公式 `∂f/∂x = 2x = 6`、`∂f/∂y = 2y = 8` 的误差在 1e-9 以内。


In [ ]:
import numpy as np
f = lambda p: p[0]**2 + p[1]**2     # f(x,y) = x^2 + y^2
h = 1e-5
p = np.array([3.0, 4.0])
dfdx = (f(p+[h,0]) - f(p-[h,0]))/(2*h)
dfdy = (f(p+[0,h]) - f(p-[0,h]))/(2*h)
print('∂f/∂x ≈', round(dfdx,3), '(真值2x=6)  ∂f/∂y ≈', round(dfdy,3), '(真值2y=8)')

## 动手观察：变化率不是一句口号

对一元函数 `f(x) = x²` 在五个点上分别用中心差分估计斜率，打印结果与解析值 `f'(x) = 2x` 对比，直观感受 h=1e-3 时的数值精度。


In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('近似斜率 =', np.round(slopes, 3))
print('理论斜率 2x =', 2 * xs)


## 代码实验：遍历不同位置，看斜率如何变化

遍历 `x ∈ [-3, 3]` 的七个点，打印每处的函数值与数值斜率，确认斜率始终接近解析导数 `f'(x) = 2x + 2`。


In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ 实现 `gradient(f, point, h=1e-5)`

对每个分量做一次中心差分，返回梯度向量。

**推理路线**：
1. 初始化 `grad = np.zeros_like(point)`，确保输出 shape 与输入相同
2. 对每个维度 i，构造扰动向量 `e`：`e = np.zeros_like(point); e[i] = h`，其余分量为 0
3. 计算 `(f(point + e) - f(point - e)) / (2 * h)`，存入 `grad[i]`
4. 关键点：每次只扰动第 i 维，其余维度不变——这正是偏导数"固定其他变量"的操作

**参考输入输出**：f(x, y) = x² + y², point = [3, 4] → ∂f/∂x = 2×3 = 6, ∂f/∂y = 2×4 = 8 → 梯度 = [6, 8]

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `gradient` 前明确三件事：
- 输入：`f`（可调用函数）、`point`（n 维 numpy 数组）、`h`（步长，默认 1e-5）
- 关键步骤：对每个维度 i 构造单位扰动向量 `e`，计算 `(f(p + e*h) - f(p - e*h)) / (2h)`
- 返回：长度为 n 的 numpy 数组，第 i 个元素是 ∂f/∂xᵢ 在 `point` 处的值


In [ ]:
def gradient(f, point, h=1e-5):
    point = np.asarray(point, dtype=float)
    grad = np.zeros_like(point)
    # ✏️ TODO: 填满 grad 的每个分量
    return grad


In [ ]:
g = gradient(lambda p: p[0]**2 + p[1]**2, [3.0, 4.0])
print('梯度 =', np.round(g, 3), '(应≈ [6, 8])')
assert np.allclose(g, [6.0, 8.0], atol=1e-2)
print('✅ 通过：你会算梯度了。')

**🔗 Aurora 连接**：训练时的损失函数（loss function） `L(W₁, W₂, …)` 是所有权重的多元函数；`backward()` 计算的正是该函数的梯度，每个权重对应梯度向量的一个分量。Aurora 实现位于 `src/aurora/audio/`；反向传播用链式法则（chain rule）代替逐参数数值扰动，把 2n 次前向传播压缩成 1 次前向 + 1 次后向，速度差距随参数量线性放大。下一课讲链式法则，解释这个压缩的来源。

In [ ]:
# 参数实验：梯度方向 = 等高线法向量
f_2d = lambda p: p[0]**2 + p[1]**2

point = [3.0, 4.0]
g = gradient(f_2d, point)
print(f'gradient(x²+y², [3,4]) = {np.round(g, 6)}')
print(f'解析值 [2×3, 2×4]      = [6, 8]')
assert np.allclose(g, [6.0, 8.0], atol=1e-4), f'梯度不对: {g}'
print('✅ 数值梯度与解析值吻合')

# 梯度方向与等高线切线正交
grad_dir = g / np.linalg.norm(g)
tangent = np.array([-g[1], g[0]])           # ∇f ⊥ 切线方向
dot = np.dot(grad_dir, tangent / np.linalg.norm(tangent))
print(f'梯度 · 切线 = {dot:.6f}（应≈0，即正交）')
assert abs(dot) < 1e-6
print('✅ 梯度与等高线正交性验证通过')


## 参数实验：梯度方向即等高线（contour line）法向量

对 f(x, y) = x² + y² 在 point = [1, 2] 调用 `gradient(f, [1.0, 2.0])`，应得到 [2, 4]（解析值：∂f/∂x = 2×1 = 2, ∂f/∂y = 2×2 = 4）。

验证梯度方向的几何含义：f(x, y) = x² + y² 的等高线是以原点为圆心的同心圆；点 (1, 2) 处的等高线切线方向是 (-2, 1)（与梯度 (2, 4) 点积为零），而梯度方向 (2, 4) 指向圆心外侧，即函数值增大最快的方向。梯度取负号得 (-2, -4)，指向圆心——这正是梯度下降每步的移动方向。

In [ ]:
# 三元函数梯度：f(x,y,z) = x² + 2y² + 3z² 在 (1,1,1)
f_3d = lambda p: p[0]**2 + 2*p[1]**2 + 3*p[2]**2
g3 = gradient(f_3d, [1.0, 1.0, 1.0])
print(f'gradient(x²+2y²+3z², [1,1,1]) = {np.round(g3, 6)}')
print(f'解析值                          = [2, 4, 6]')
assert np.allclose(g3, [2.0, 4.0, 6.0], atol=1e-4)
print('✅ 三元函数梯度验证通过')
print(f'梯度范数 = {np.linalg.norm(g3):.4f}（下降最陡方向的步长缩放因子）')


## 本课收束

现在可以调用 `gradient(f, point)` 对任意多元函数在任意点求数值梯度（numerical gradient），返回与 `point` 等长的偏导向量。梯度的每一维对应一个输入变量，指向使 `f` 增大最快的方向；训练时取负号即得下降方向。Aurora 的 `backward` 计算的是同一个向量，只是用解析推导代替数值扰动，速度快几个数量级。

下一课：**L24**（链式法则）讲复合函数的梯度为何等于各层偏导的乘积。